# 24. Rolling60-day full-feature trajectory benchmark

60-day rolling training interval, re-estimation every7 days, daily forecasts of future1/2/3-day raw new onset counts. Initial training March1-April29 2020. Development April30-December31 2020; subsequent evaluation2021-2023 after freezing settings. Same585 ZIP cohort and all34 candidate columns including Pressure. No feature selection. Each training target must have matured by fit date. Predictors update daily. This replaces the fixed-window protocol for the NEW experiment only; Notebooks21-23 remain intact.

A 60-calendar-day window contains14 warmup days, leaving45/44/43 training origins for horizons1/2/3. All3 forecast targets must stay within their phase, so final forecast origins are December28. This is retrospective exploratory evaluation, not an untouched prospective test.

In [1]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
GROUPS={
 'core_history':['cases_today','cases_lag_1','cases_lag_7','cases_lag_14'],
 'rolling_level':['cases_mean_7d','cases_mean_14d'],
 'volatility':['case_std_7d','case_std_14d'],
 'dynamics':['case_acceleration','case_jerk'],
 'local_rates':['case_rate_today_per_100k','case_rate_7d_per_100k','case_rate_14d_per_100k'],
 'sparse_history':['days_since_last_observed_case_censored','no_previous_observed_case'],
 'pressure':['nj_covid_pressure_7d','nj_covid_pressure_14d'],
 'month':[f'target_month_{k}' for k in range(2,13)],
 'weekday':[f'target_weekday_{k}' for k in range(2,8)]}

def run_rolling60(source, cohort_file, output, stage='development', evaluation_plan_frozen=False):
    if stage not in ['development','evaluation']: raise ValueError('Unknown stage')
    if stage=='evaluation' and not evaluation_plan_frozen:
        raise ValueError('Freeze model/features/hyperparameters before running subsequent evaluation.')
    source,output=Path(source),Path(output)
    output.mkdir(parents=True,exist_ok=True)
    with source.open('rb') as f:
        if f.read(7)==b'version': raise ValueError('Input is a Git LFS pointer. Download the real gzip data first.')
    ids=pd.read_csv(cohort_file,dtype={'zip':str})['zip'].to_numpy(dtype='U5')
    numeric=sum([v for k,v in GROUPS.items() if k not in ['month','weekday','sparse_history']],[])
    cols=['zip','date','pressure_definition']+numeric
    frame=pd.read_csv(source,usecols=cols,dtype={'zip':str},parse_dates=['date'])
    dates=pd.date_range('2020-03-01','2023-12-31')
    frame=frame.loc[frame.zip.isin(ids)&frame.date.isin(dates)]
    assert len(ids)==585 and not frame.duplicated(['zip','date']).any()
    assert frame.pressure_definition.eq('valid_population_nj_origins_renormalized_v2').all()
    arrays={c:frame.pivot(index='zip',columns='date',values=c).reindex(index=ids,columns=dates).to_numpy(float) for c in numeric}
    counts=arrays['cases_today'];features=sum(GROUPS.values(),[])
    assert np.isfinite(counts).all() and (counts>=0).all()
    # End-of-day origins: initial fit includes mature targets through April29.
    # A 60-day interval includes14 warmup days, leaving45/44/43 pairs per ZIP.
    first=int(dates.get_loc('2020-04-29'))
    origin_start='2020-04-30' if stage=='development' else '2021-01-01'
    origin_end='2020-12-28' if stage=='development' else '2023-12-28'
    origins=np.flatnonzero((dates>=origin_start)&(dates<=origin_end))
    actual=np.stack([counts[:,origins+h] for h in (1,2,3)],axis=2)
    pred=np.full_like(actual,np.nan);logs=[];coefficients=[];omitted=[]
    # Anchor weekly schedule to April29, keep same schedule across stage boundaries.
    fit_indices=np.unique(first+7*((origins-first)//7))
    for fit in fit_indices:
        lo=fit-59
        batch=np.flatnonzero((origins>=fit)&(origins<fit+7))
        assert lo>=0 and len(batch)>0
        last=np.full(len(ids),-1,dtype=int)
        gap=np.empty((len(ids),len(dates)));never=np.empty_like(gap)
        gap[:]=np.nan;never[:]=np.nan
        for t in range(lo,max(fit,int(origins[batch[-1]]))+1):
            last=np.where(counts[:,t]>0,t,last)
            never[:,t]=(last<0);gap[:,t]=np.where(last>=0,t-last,t-lo+1)
        arrays['no_previous_observed_case']=never
        arrays['days_since_last_observed_case_censored']=gap
        for hh,h in enumerate((1,2,3)):
            train=np.arange(lo+14,fit-h+1)
            target=dates+pd.Timedelta(days=h)
            for k in range(2,13): arrays[f'target_month_{k}']=np.broadcast_to((target.month==k).astype(float),counts.shape)
            for k in range(2,8): arrays[f'target_weekday_{k}']=np.broadcast_to((target.dayofweek==k-1).astype(float),counts.shape)
            xt=np.stack([arrays[f][:,train] for f in features],axis=2)
            xp=np.stack([arrays[f][:,origins[batch]] for f in features],axis=2)
            yt=counts[:,train+h]
            assert np.isfinite(xt).all() and np.isfinite(xp).all()
            assert train[0]-14>=lo and train[-1]+h<=fit and np.all(origins[batch]+h>fit)
            xm=xt.mean(axis=1);ym=yt.mean(axis=1)
            xc=xt-xm[:,None];yc=yt-ym[:,None]
            norms=np.sqrt((xc*xc).mean(axis=(0,1)));scale=np.where(norms>1e-10,norms,1)
            z=(xc/scale).reshape(-1,len(features));keep=[];q=[]
            for j,f in enumerate(features):
                if norms[j]<1e-10:
                    omitted.append(dict(fit_date=str(dates[fit].date()),horizon=h,feature=f,reason='Training-only within-ZIP constant'));continue
                v=z[:,j].copy()
                for _ in range(2):
                    for u in q: v-=u*(u@v)
                length=np.linalg.norm(v)
                if length<=1e-7*np.sqrt(len(v)):
                    omitted.append(dict(fit_date=str(dates[fit].date()),horizon=h,feature=f,reason='Training-only exact linear dependence'));continue
                keep.append(j);q.append(v/length)
            b=np.linalg.lstsq(z[:,keep],yc.ravel(),rcond=None)[0]/scale[keep]
            alpha=ym-xm[:,keep]@b
            pred[:,batch,hh]=np.einsum('ntp,p->nt',xp[:,:,keep],b)+alpha[:,None]
            for j,value in zip(keep,b):coefficients.append(dict(fit_date=str(dates[fit].date()),horizon=h,feature=features[j],coefficient=float(value)))
            logs.append(dict(fit_date=str(dates[fit].date()),window_start=str(dates[lo].date()),window_end=str(dates[fit].date()),window_calendar_days=60,horizon=h,training_origin_start=str(dates[train[0]].date()),training_origin_end=str(dates[train[-1]].date()),latest_training_target=str(dates[train[-1]+h].date()),usable_training_origins=len(train),forecast_origin_start=str(dates[origins[batch[0]]].date()),forecast_origin_end=str(dates[origins[batch[-1]]].date()),candidate_columns=len(features),identifiable_slopes=len(keep)))
        print('Completed fit:',dates[fit].date(),flush=True)
    assert np.isfinite(pred).all()
    np.savez_compressed(output/'predictions.npz',raw_predictions=pred,actual_cases=actual,zip=ids,forecast_origins=dates[origins].to_numpy(),horizons=[1,2,3])
    pd.DataFrame(logs).to_csv(output/'training_timing.csv',index=False)
    pd.DataFrame(coefficients).to_csv(output/'coefficients.csv',index=False)
    pd.DataFrame(omitted).to_csv(output/'unestimable_feature_log.csv',index=False)
    pd.DataFrame([(g,f) for g,fs in GROUPS.items() for f in fs],columns=['group','feature']).to_csv(output/'candidate_features.csv',index=False)
    p=np.maximum(pred,0);err=p-actual;da=np.diff(actual,axis=2);dp=np.diff(p,axis=2)
    aa=np.diff(actual,n=2,axis=2)[...,0];ap=np.diff(p,n=2,axis=2)[...,0]
    daily=pd.DataFrame(dict(forecast_origin=dates[origins],trajectory_MAE=np.abs(err).mean(axis=(0,2)),MSE=(err**2).mean(axis=(0,2)),mean_bias=err.mean(axis=(0,2)),slope_MAE=np.abs((p[...,2]-p[...,0]-actual[...,2]+actual[...,0])/2).mean(axis=0),direction_agreement=(np.sign(p[...,2]-p[...,0])==np.sign(actual[...,2]-actual[...,0])).mean(axis=0),two_step_shape_agreement=(np.sign(dp)==np.sign(da)).all(axis=2).mean(axis=0),change_MAE=np.abs(dp-da).mean(axis=(0,2)),acceleration_MAE=np.abs(ap-aa).mean(axis=0),all_zero_actual_path_share=(actual==0).all(axis=2).mean(axis=0)))
    daily.to_csv(output/'scores_by_forecast_origin.csv',index=False)
    rows=[]
    periods={'overall':np.repeat('all',len(daily)),'year':daily.forecast_origin.dt.year.astype(str).to_numpy(),'quarter':daily.forecast_origin.dt.to_period('Q').astype(str).to_numpy()}
    for scope,values in periods.items():
        for label in np.unique(values):
            d=daily.loc[values==label];r=dict(scope=scope,period=label,forecast_origins=len(d),regions=len(ids),paths=len(d)*len(ids),RMSE=float(np.sqrt(d.MSE.mean())))
            r.update({col:float(d[col].mean()) for col in daily.columns if col not in ['forecast_origin','MSE']});rows.append(r)
    scores=pd.DataFrame(rows);scores.to_csv(output/'trajectory_scores_by_period.csv',index=False)
    protocol=dict(stage=stage,initial_training=['2020-03-01','2020-04-29'],development=['2020-04-30','2020-12-31'],evaluation=['2021-01-01','2023-12-31'],window_calendar_days=60,retrain_every_days=7,first_fit='2020-04-29 end of day',prediction='Daily end-of-day origin predicts t+1,t+2,t+3; raw onset counts',boundaries='All3 targets remain within stage; development origins end Dec28 2020, evaluation origins end Dec28 2023. December29-31 origins omitted to avoid crossing targets.',maturity='Every training target <=fit date; forecast targets >forecast origin>=fit date',feature_policy='All34 candidates including bothPressure; ZIPFE absorbs static attributes; constants/dependencies logged using training only',warmup='14 days inside each60-day window,45/44/43 usable training origins per ZIP per horizon; sparse-history state reset at window start',inputs='Updated daily through origin; coefficients updated only on anchored7-day schedule',evaluation_policy='Algorithm and settings frozen before subsequent-period run. Past realized evaluation labels may enter future scheduled fits; no retrospective prediction changes or manual tuning.',limitation='Exploratory retrospective time evaluation;2021-2023 previously inspected. Onset-date availability assumed, reporting delays not modeled.',source_sha256=hashlib.sha256(source.read_bytes()).hexdigest())
    (output/'protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf8')
    (output/'verification.json').write_text(json.dumps(dict(all_training_targets_mature=True,window_exactly60=True,forecast_targets_after_fit=True,all_predictions_finite=True,all34_candidates_considered=True,per_zip_metrics_produced=False),indent=2))
    (output/'RESULTS.md').write_text('# Rolling60-day trajectory evaluation\n\n'+scores.to_string(index=False)+'\n\n60 calendar days,14 within-window warmup days;45/44/43 supervised origins per horizon per ZIP. Weekly re-estimation, daily predictor updates. No Pressure ablation or seasonal naive. Primary score is ZIP-path trajectory MAE; change/acceleration and direction/shape are supplementary. Scores are retrospective and exploratory. See protocol.json and training_timing.csv for information boundaries. No coefficient p-values are asserted for this rolling dynamic-FE estimator.\n',encoding='utf8')
    return scores, output


In [2]:
ROOT = Path.cwd()
if not (ROOT / 'fixed60_trajectory_outputs').exists() and (ROOT / 'Weikun').is_dir():
    ROOT = ROOT / 'Weikun'
SOURCE = ROOT / 'rolling_trajectory_outputs' / 'inputs' / 'nj_daily_features.csv.gz'
COHORT = ROOT / 'fixed60_trajectory_outputs' / 'fixed60_20261027' / 'common_regions.csv'
OUTPUT = ROOT / 'rolling_trajectory_outputs' / 'rolling60_weekly_development_20261027'
# SOURCE must contain actual data, not a Git LFS pointer.
scores, folder = run_rolling60(SOURCE, COHORT, OUTPUT, stage='development')
display(scores)
print(folder.resolve())

Completed fit: 2020-04-29
Completed fit: 2020-05-06
Completed fit: 2020-05-13
Completed fit: 2020-05-20
Completed fit: 2020-05-27
Completed fit: 2020-06-03
Completed fit: 2020-06-10
Completed fit: 2020-06-17
Completed fit: 2020-06-24
Completed fit: 2020-07-01
Completed fit: 2020-07-08
Completed fit: 2020-07-15
Completed fit: 2020-07-22
Completed fit: 2020-07-29
Completed fit: 2020-08-05
Completed fit: 2020-08-12
Completed fit: 2020-08-19
Completed fit: 2020-08-26
Completed fit: 2020-09-02
Completed fit: 2020-09-09
Completed fit: 2020-09-16
Completed fit: 2020-09-23
Completed fit: 2020-09-30
Completed fit: 2020-10-07
Completed fit: 2020-10-14
Completed fit: 2020-10-21
Completed fit: 2020-10-28
Completed fit: 2020-11-04
Completed fit: 2020-11-11
Completed fit: 2020-11-18
Completed fit: 2020-11-25
Completed fit: 2020-12-02
Completed fit: 2020-12-09
Completed fit: 2020-12-16
Completed fit: 2020-12-23


,scope,period,forecast_origins,regions,paths,RMSE,trajectory_MAE,mean_bias,slope_MAE,direction_agreement,two_step_shape_agreement,change_MAE,acceleration_MAE,all_zero_actual_path_share
0,overall,all,243,585,142155,1.418396,0.774283,0.067025,0.455175,0.281552,0.126116,0.896647,1.589743,0.428145
1,year,2020,243,585,142155,1.418396,0.774283,0.067025,0.455175,0.281552,0.126116,0.896647,1.589743,0.428145
2,quarter,2020Q2,62,585,36270,1.040407,0.498861,0.118157,0.283829,0.278991,0.145547,0.565715,1.021620,0.575903
3,quarter,2020Q3,92,585,53820,0.772692,0.426897,0.009563,0.241424,0.206447,0.089521,0.469918,0.864522,0.552861
4,quarter,2020Q4,89,585,52065,2.030217,1.325246,0.090802,0.795497,0.360972,0.150408,1.568296,2.735180,0.196293


C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun\rolling_trajectory_outputs\rolling60_weekly_development_20261027


## Subsequent2021-2023 evaluation — intentionally not automatic
Freeze model definitions, features and settings first. This remains retrospective evaluation because portions of later years have already been inspected. Scheduled refits may use earlier realized labels, never targets after fit. Do not tune manually on these results.

In [3]:
RUN_SUBSEQUENT_EVALUATION = False
if RUN_SUBSEQUENT_EVALUATION:
    evaluation_scores, evaluation_folder = run_rolling60(SOURCE, COHORT, ROOT / 'rolling_trajectory_outputs' / 'rolling60_weekly_evaluation_20261027', stage='evaluation', evaluation_plan_frozen=True)
    display(evaluation_scores)